# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

The reason I picked a line chart is that it clearly depicts changes in population growth over time in the various continents. To ensure an equal basis for comparison, I have standardized all continents in terms of population index, which is pegged to 100 at year 1970. I have differentiated the continents using colour codes and noted the most rapidly growing continent.

In [2]:
# Q1

continent_year = (
    df_long.groupby(['Continent', 'Year'], as_index=False)['Population']
    .sum()
)

base = (
    continent_year[continent_year['Year'] == 1970]
    [['Continent', 'Population']]
    .rename(columns={'Population': 'Base Population'})
)

continent_year = continent_year.merge(base, on='Continent')
continent_year['Growth Index'] = (
    continent_year['Population'] / continent_year['Base Population'] * 100
)

latest = continent_year[continent_year['Year'] == 2022]
fastest = latest.loc[latest['Growth Index'].idxmax(), 'Continent']

fig = px.line(
    continent_year,
    x='Year',
    y='Growth Index',
    color='Continent',
    markers=True,
    title=f'{fastest} Has Experienced the Fastest Relative Population Growth',
    labels={'Growth Index': 'Population index (1970 = 100)'}
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False),
    legend_title_text='Continent'
)

fig.show()


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

My choice for displaying a chart is a horizontal bar chart in sorted form since this makes it easier to compare the extra population of the top 15 nations. Sorting the bars in order from high to low enables us to see which are the largest contributors right away. The bars representing Africa were done in Purple while the rest in gold.

In [12]:
# Q2

df['Population Added'] = (
    df['2022 Population'] - df['1970 Population']
)

top15 = df.nlargest(15, 'Population Added').copy()
top15 = top15.sort_values('Population Added', ascending=True)

africa_count = (top15['Continent'] == 'Africa').sum()

top15['Region Highlight'] = np.where(
    top15['Continent'] == 'Africa',
    'Africa',
    'Other Continents'
)

fig = px.bar(
    top15,
    x='Population Added',
    y='Country',
    orientation='h',
    color='Region Highlight',
    color_discrete_map={
        'Africa': '#8E5EA2',
        'Other Continents': '#D9A441'
    },
    text='Population Added',
    title=f'Top 15 Countries by Population Added (1970–2022) — {africa_count} from Africa',
    labels={'Population Added': 'Additional people', 'Country': ''}
)

fig.update_traces(
    texttemplate='%{x:,.0f}',
    textposition='outside',
    cliponaxis=False
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    showlegend=True,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False),
    margin=dict(r=100)
)

fig.show()

print("Number of African countries in the top 15:", africa_count)


Number of African countries in the top 15: 4


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I have selected a scatter plot which is the best choice for visualizing the correlation between the land area and population size of the countries individually. The continents are distinguished by colors, while the logarithmic axis allows comparison of different countries which have a large difference in size and population.

In [13]:

# Q3

fig = px.scatter(
    df,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    log_x=True,
    log_y=True,
    color_discrete_sequence=px.colors.qualitative.Safe,
    opacity=0.75,
    title='Land area and population are related, but not proportionally'
)

fig.update_traces(
    marker=dict(size=9, line=dict(width=0.5, color='white'))
)

fig.update_layout(
    template='plotly_white',
    xaxis_title='Land area (km², log scale)',
    yaxis_title='2022 population (log scale)',
    legend_title='Continent',
    title_x=0.5
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='lightgrey')

fig.show()


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I have selected a violin plot because it shows the distribution of population density within each continent rather than only displaying the average. The width of each violin represents how frequently density values occur, making it easier to identify variation and concentration. I used a logarithmic vertical axis because population density differs considerably between countries.

In [17]:

# Q4

fig = px.violin(
    df,
    x='Continent',
    y='Density',
    color='Continent',
    box=True,
    points='outliers',
    log_y=True,
    title='Population Density Varies Widely Within Each Continent',
    labels={
        'Continent': '',
        'Density': 'People per km² (log scale)'
    }
)

fig.update_traces(
    meanline_visible=True,
    opacity=0.8
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    showlegend=False,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False)
)

fig.show()


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I have selected a choropleth map because it enables comparison of population changes geographically around the world. The colors in each country depend on the country’s percentage change in population from 2000 to 2022. The use of a divergent color scheme that ranges from blue to orange helps identify whether there was an increase or decrease in population.

In [6]:

# Q5

df['Population Change (%)'] = (
    (df['2022 Population'] - df['2000 Population'])
    / df['2000 Population'] * 100
)

max_change = df['Population Change (%)'].abs().max()

fig = px.choropleth(
    df,
    locations='CCA3',
    locationmode='ISO-3',
    color='Population Change (%)',
    hover_name='Country',
    hover_data={
        'Population Change (%)': ':.2f',
        'CCA3': False,
        '2022 Population': ':,'
    },
    color_continuous_scale=[
        [0, '#2878B5'],
        [0.5, '#F0F0F0'],
        [1, '#D98236']
    ],
    range_color=[-max_change, max_change],
    title='Population Growth and Decline Across the World (2000–2022)',
    labels={'Population Change (%)': 'Population change (%)'}
)

fig.update_geos(
    showframe=False,
    showcoastlines=False,
    showland=True,
    landcolor='#E5E5E5',
    showcountries=True
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    geo=dict(bgcolor='white'),
    margin=dict(l=0, r=0, t=60, b=0)
)

fig.show()

print("Countries with population growth:",
      (df['Population Change (%)'] > 0).sum())

print("Countries with population decline:",
      (df['Population Change (%)'] < 0).sum())


Countries with population growth: 200
Countries with population decline: 34


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I have selected a treemap because it displays hierarchical population distribution in a single visualization. The larger rectangles represent continents with larger populations, while the smaller rectangles inside each continent represent individual countries. This makes it possible to compare both continental and country-level population contributions without using separate charts.

In [18]:
# Q6

fig = px.treemap(
    df,
    path=['Continent', 'Country'],
    values='2022 Population',
    color='Continent',
    color_discrete_map={
        'Asia': '#4E79A7',
        'Africa': '#F28E2B',
        'Europe': '#59A14F',
        'North America': '#B07AA1',
        'South America': '#E15759',
        'Oceania': '#76B7B2'
    },
    title='Asia and Africa account for large shares of the world population'
)

fig.update_traces(
    textinfo='label+percent parent',
    textfont_size=13,
    marker=dict(line=dict(color='white', width=2))
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    margin=dict(t=60, l=10, r=10, b=10)
)

fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I have selected a stacked area chart because it clearly shows how the share of world population held by each region changes between 1970 and 2022. Different colours distinguish the four regions, while the total area represents 100% of the world population.

In [20]:
# Q7 - Whose share of the world is rising?

# Step 1: Group the other three continents
df_long['Region'] = df_long['Continent'].replace({
    'North America': 'Rest of the world',
    'South America': 'Rest of the world',
    'Oceania': 'Rest of the world'
})

# Step 2: Calculate the total population by region and year
continent_share = (
    df_long.groupby(['Year', 'Region'])['Population']
    .sum()
    .reset_index()
)

# Step 3: Calculate the total world population for each year
world_total = (
    continent_share.groupby('Year')['Population']
    .sum()
    .reset_index(name='World Population')
)

# Step 4: Calculate each region's share of world population
continent_share = continent_share.merge(world_total, on='Year')

continent_share['Share'] = (
    continent_share['Population'] / continent_share['World Population']
)

# Step 5: Create the stacked area chart
fig = px.area(
    continent_share,
    x='Year',
    y='Share',
    color='Region',
    category_orders={
        'Region': ['Africa', 'Asia', 'Europe', 'Rest of the world']
    },
    color_discrete_map={
        'Africa': '#0072B2',
        'Asia': '#E69F00',
        'Europe': '#CC79A7',
        'Rest of the world': '#999999'
    },
    title='Africa’s share is rising while Europe’s is declining'
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    xaxis_title='Year',
    yaxis_title='Share of world population',
    yaxis_tickformat='.0%',
    legend_title='Region'
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='lightgrey')

fig.show()